In [404]:
# Cell 0 — Project Imports

import torch
from torch.nn import functional as F

In [405]:
# Cell 1 — 중심 좌표로 3D Patch 자르기

def crop_3d_patch(
    volume: torch.Tensor,                 # [B, C, D, H, W]    
    center_zyx: tuple[int, int, int],     # (center_z, center_y, center_x)
    patch_size_zyx: tuple[int, int, int], # (patch_D, patch_H, patch_W)
) -> torch.Tensor:                        # [B, C, patch_D, patch_H, patch_W]
    """Volume 내부의 중심 좌표를 기준으로 3D patch를 자른다."""
    
    if volume.ndim != 5:
        raise ValueError(
            "volume shape은 [B, C, D, H, W]여야 합니다."
        )
        
    center_z, center_y, center_x = center_zyx
    patch_depth, patch_height, patch_width = patch_size_zyx
    
    # 중심 좌표에서 patch 크기의 절반만큼 뒤로 이동
    start_z = center_z - patch_depth // 2
    start_y = center_y - patch_height // 2
    start_x = center_x - patch_width // 2

    # Python slicing의 끝 index는 포함되지 않음
    end_z = start_z + patch_depth
    end_y = start_y + patch_height
    end_x = start_x + patch_width
    
    volume_depth, volume_height, volume_width = volume.shape[-3:]
    
    
    patch_is_outside = (
        start_z < 0
        or start_y < 0
        or start_x < 0
        or end_z > volume_depth
        or end_y > volume_height
        or end_x > volume_width
    )
    
    if patch_is_outside:
        raise ValueError(
            "요청한 patch가 volume 경계를 벗어났습니다."
        )
        
        
    patch = volume[
        :,
        :,
        start_z:end_z,
        start_y:end_y,
        start_x:end_x,
    ]
    
    return patch



synthetic_volume = torch.arange(
    1 * 1 * 8 * 10 * 12,
    dtype=torch.float32,
).reshape(
    1,
    1,
    8,
    10,
    12,
)

patch_center = (
    4,
    5,
    6,
)

patch_size = (
    4,
    6,
    8,
)

cropped_patch = crop_3d_patch(
    volume=synthetic_volume,
    center_zyx=patch_center,
    patch_size_zyx=patch_size,
)

print("Volume shape:     ", synthetic_volume.shape)
print("Center (z, y, x): ", patch_center)
print("Patch size:       ", patch_size)
print("Patch shape:      ", cropped_patch.shape)
print("First value:      ", cropped_patch[0, 0, 0, 0, 0].item())
print("Last value:       ", cropped_patch[0, 0, -1, -1, -1].item())

Volume shape:      torch.Size([1, 1, 8, 10, 12])
Center (z, y, x):  (4, 5, 6)
Patch size:        (4, 6, 8)
Patch shape:       torch.Size([1, 1, 4, 6, 8])
First value:       266.0
Last value:        693.0


In [406]:
# Cell 2 — Volume 경계 Patch에 Padding 적용

def crop_3d_patch_with_padding(
    volume: torch.Tensor,                  # [B, C, D, H, W]
    center_zyx: tuple[int, int, int],      # (center_z, center_y, center_x)
    patch_size_zyx: tuple[int, int, int],  # (patch_D, patch_H, patch_W)
    padding_value: float,
) -> torch.Tensor:                         # [B, C, patch_D, patch_H, patch_W]
    """Volume 경계를 벗어난 영역을 padding하여 고정 크기 patch를 반환"""
    
    if volume.ndim != 5:
        raise ValueError(
            "volume shape은 [B, C, D, H, W]여야 합니다."
        )
        
    center_z, center_y, center_x = center_zyx
    patch_depth, patch_height, patch_width = patch_size_zyx

    start_z = center_z - patch_depth // 2
    start_y = center_y - patch_height // 2
    start_x = center_x - patch_width // 2

    end_z = start_z + patch_depth
    end_y = start_y + patch_height
    end_x = start_x + patch_width
    
    volume_depth, volume_height, volume_width = volume.shape[-3:]
    
    
    # 실제 volume 내부에서 읽을 수 있는 범위만 남김
    valid_start_z = max(start_z, 0)
    valid_start_y = max(start_y, 0)
    valid_start_x = max(start_x, 0)

    valid_end_z = min(end_z, volume_depth)
    valid_end_y = min(end_y, volume_height)
    valid_end_x = min(end_x, volume_width)
    
    cropped_region = volume[
        :,
        :,
        valid_start_z:valid_end_z,
        valid_start_y:valid_end_y,
        valid_start_x:valid_end_x,
    ]
    
    
    # 요청 범위가 volume을 얼마나 벗어났는지 계산
    padding_front_z = max(0, -start_z)
    padding_back_z = max(0, end_z - volume_depth)

    padding_top_y = max(0, -start_y)
    padding_bottom_y = max(0, end_y - volume_height)

    padding_left_x = max(0, -start_x)
    padding_right_x = max(0, end_x - volume_width)
    
    
    # 부족한 영역을 padding_value로 채움
    # (x_left, x_right, y_top, y_bottom, z_front, z_back)
    padded_patch = F.pad(
        cropped_region,
        pad=(
            padding_left_x,
            padding_right_x,
            padding_top_y,
            padding_bottom_y,
            padding_front_z,
            padding_back_z,
        ),
        mode="constant",
        value=padding_value,
    )

    # [B, C, patch_D, patch_H, patch_W]
    return padded_patch



boundary_center = (
    0,
    1,
    1,
)

boundary_patch = crop_3d_patch_with_padding(
    volume=synthetic_volume,
    center_zyx=boundary_center,
    patch_size_zyx=patch_size,
    padding_value=-1.0,
)

padding_voxel_count = (
    boundary_patch == -1.0
).sum()

print("Volume shape:       ", synthetic_volume.shape)
print("Boundary center:    ", boundary_center)
print("Requested size:     ", patch_size)
print("Result shape:       ", boundary_patch.shape)
print("Padding voxel count:", padding_voxel_count.item())
print("Corner value:       ", boundary_patch[0, 0, 0, 0, 0].item())
print("Center value:       ", boundary_patch[0, 0, 2, 3, 4].item())

Volume shape:        torch.Size([1, 1, 8, 10, 12])
Boundary center:     (0, 1, 1)
Requested size:      (4, 6, 8)
Result shape:        torch.Size([1, 1, 4, 6, 8])
Padding voxel count: 152
Corner value:        -1.0
Center value:        13.0


In [407]:
# Cell 3 — Uniform Patch-Center Sampling

def sample_uniform_center(
    volume_shape_zyx: tuple[int, int, int],  # (D, H, W)
    generator: torch.Generator,
) -> tuple[int, int, int]:                   # (center_z, center_y, center_x)
    """Volume의 모든 voxel 중 하나를 균일한 확률로 선택"""
    
    volume_depth, volume_height, volume_width = volume_shape_zyx
    
    # 0 ~ volume_depth-1 사이 정수 하나 랜덤 선택
    center_z = torch.randint(
        low=0,
        high=volume_depth,
        size=(1,),
        generator=generator,
    ).item()

    center_y = torch.randint(
        low=0,
        high=volume_height,
        size=(1,),
        generator=generator,
    ).item()

    center_x = torch.randint(
        low=0,
        high=volume_width,
        size=(1,),
        generator=generator,
    ).item()

    return center_z, center_y, center_x


# Label contract: [B, C=1, D, H, W] / dtype=torch.long
synthetic_label = torch.zeros(
    1,
    1,
    32,
    64,
    64,
    dtype=torch.long,
)

# 작은 가상 장기 class 1을 volume 내부에 배치
synthetic_label[
    :,
    :,
    14:18,
    28:36,
    28:36,
] = 1
    
    
sampling_generator = torch.Generator()
sampling_generator.seed()

uniform_patch_size = (
    8,
    16,
    16,
)

number_of_samples = 10
foreground_hit_count = 0

for sample_index in range(number_of_samples):
    
    # volume 전체에서 랜덤 center 좌표 하나 선택
    sampled_center = sample_uniform_center(
        volume_shape_zyx=synthetic_label.shape[-3:],
        generator=sampling_generator,
    )

    # 랜덤 center를 기준으로 3D patch 추출 (경계를 넘어가면 0으로 padding)
    sampled_label_patch = crop_3d_patch_with_padding(
        volume=synthetic_label,             # [B, 1, D, H, W]
        center_zyx=sampled_center,
        patch_size_zyx=uniform_patch_size,
        padding_value=0.0,
    )                                      

    # 뽑힌 patch 안에 class 1 voxel이 몇 개 있는지 계산
    foreground_voxel_count = (
        sampled_label_patch == 1
    ).sum().item()

    # class 1 voxel이 하나라도 있으면
    # 이 patch는 foreground(장기)를 포함한 것으로 판단
    contains_foreground = (
        foreground_voxel_count > 0
    )

    # 장기를 포함한 patch 개수 증가
    if contains_foreground:
        foreground_hit_count += 1

    print(
        f"Sample {sample_index:2d} | "
        f"center={sampled_center} | "
        f"foreground voxels={foreground_voxel_count}"
    )

print()
print("Label shape:          ", synthetic_label.shape)
print("Patch size:           ", uniform_patch_size)
print("Foreground hit count:", foreground_hit_count)
print("Total samples:        ", number_of_samples)

Sample  0 | center=(22, 30, 4) | foreground voxels=0
Sample  1 | center=(0, 44, 22) | foreground voxels=0
Sample  2 | center=(5, 57, 30) | foreground voxels=0
Sample  3 | center=(5, 29, 30) | foreground voxels=0
Sample  4 | center=(14, 9, 52) | foreground voxels=0
Sample  5 | center=(6, 43, 14) | foreground voxels=0
Sample  6 | center=(27, 34, 21) | foreground voxels=0
Sample  7 | center=(11, 61, 41) | foreground voxels=0
Sample  8 | center=(17, 33, 34) | foreground voxels=256
Sample  9 | center=(23, 0, 21) | foreground voxels=0

Label shape:           torch.Size([1, 1, 32, 64, 64])
Patch size:            (8, 16, 16)
Foreground hit count: 1
Total samples:         10


In [408]:
# Cell 4 — Foreground Patch-Center Sampling

# Uniform sampling
# → 아무 위치나 center
# → background가 많이 뽑힘

# Foreground sampling
# → 장기 내부 위치만 center
# → 장기를 무조건 포함하는 patch를 뽑음

def sample_foreground_center(
    label_volume: torch.Tensor,          # [B=1, C=1, D, H, W]
    foreground_class: int,               # class ID (0 ~ 9)
    generator: torch.Generator,
) -> tuple[int, int, int]:               # (center_z, center_y, center_x)
    """지정한 foreground class의 voxel 중 하나를 patch center로 선택"""

    if label_volume.ndim != 5:
        raise ValueError(
            "label_volume shape은 [B, C, D, H, W] 여야 함."
        )

    if label_volume.shape[0] != 1 or label_volume.shape[1] != 1:
        raise ValueError(
            "현재 sampler는 [B=1, C=1, D, H, W] label을 사용"
        )
        
    # Batch와 channel 차원을 제거하고 class mask를 만든다.
    foreground_mask = (
        label_volume[0, 0] == foreground_class
    )
    
    # True인 모든 foreground voxel의 좌표를 추출
    # [N_foreground, 3]
    foreground_coordinates = torch.nonzero(
        foreground_mask,
        as_tuple=False,
    )                                   

    number_of_foreground_voxels = (
        foreground_coordinates.shape[0]
    )
    
    if number_of_foreground_voxels == 0:
        raise ValueError(
            f"Class {foreground_class} voxel이 존재하지 않음."
        )

    # N개의 foreground 좌표 중 하나의 row index를 균일하게 선택
    sampled_coordinate_index = torch.randint(
        low=0,
        high=number_of_foreground_voxels,
        size=(1,),
        generator=generator,
    ).item()

    # 선택된 row에는 (z, y, x) 좌표가 들어 있다.
    # [3]
    sampled_coordinate = foreground_coordinates[
        sampled_coordinate_index
    ]                                      

    center_z, center_y, center_x = (
        sampled_coordinate.tolist()
    )

    return center_z, center_y, center_x



foreground_generator = torch.Generator()
foreground_seed = foreground_generator.seed()

print("Foreground sampling seed:", foreground_seed)
print()

number_of_foreground_samples = 10
foreground_center_hit_count = 0

for sample_index in range(number_of_foreground_samples):
    
    # Class 1 voxel 중 하나를 patch center로 선택한다.
    foreground_center = sample_foreground_center(
        label_volume=synthetic_label,       # [1, 1, 32, 64, 64]
        foreground_class=1,
        generator=foreground_generator,
    )

    center_z, center_y, center_x = foreground_center

    # 선택한 center가 실제 class 1인지 확인한다.
    center_class = synthetic_label[
        0,
        0,
        center_z,
        center_y,
        center_x,
    ].item()

    # Foreground center를 기준으로 고정 크기 label patch를 추출한다.
    foreground_label_patch = crop_3d_patch_with_padding(
        volume=synthetic_label,             # [B, 1, D, H, W]
        center_zyx=foreground_center,
        patch_size_zyx=uniform_patch_size,
        padding_value=0.0,
    )                                       # [B, 1, patch_D, patch_H, patch_W]

    # 추출한 patch 안에 포함된 class 1 voxel 수를 계산한다.
    foreground_voxel_count = (
        foreground_label_patch == 1
    ).sum().item()

    if center_class == 1:
        foreground_center_hit_count += 1

    print(
        f"Sample {sample_index:2d} | "
        f"center={foreground_center} | "
        f"center class={center_class} | "
        f"foreground voxels={foreground_voxel_count}"
    )

print()
print(
    "Foreground center hit count:",
    foreground_center_hit_count,
)
print(
    "Total samples:               ",
    number_of_foreground_samples,
)

Foreground sampling seed: 9489910964568772156

Sample  0 | center=(16, 30, 35) | center class=1 | foreground voxels=256
Sample  1 | center=(16, 31, 28) | center class=1 | foreground voxels=256
Sample  2 | center=(16, 34, 33) | center class=1 | foreground voxels=256
Sample  3 | center=(16, 31, 32) | center class=1 | foreground voxels=256
Sample  4 | center=(17, 33, 35) | center class=1 | foreground voxels=256
Sample  5 | center=(14, 29, 31) | center class=1 | foreground voxels=256
Sample  6 | center=(16, 33, 29) | center class=1 | foreground voxels=256
Sample  7 | center=(15, 34, 32) | center class=1 | foreground voxels=256
Sample  8 | center=(14, 34, 28) | center class=1 | foreground voxels=256
Sample  9 | center=(17, 29, 33) | center class=1 | foreground voxels=256

Foreground center hit count: 10
Total samples:                10


In [ ]:
# Cell 5 — Uniform과 Foreground Sampling 비교

# Uniform
# → 대부분 background patch

# Foreground
# → 모든 patch가 장기 포함

# OLES3D
# → 장기 포함 여부를 넘어
#   어떤 장기 × 어떤 오류 위치를 노출할지 결정

def measure_foreground_patch_hit_rate(
    label_volume: torch.Tensor,             # [B=1, C=1, D, H, W]
    patch_size_zyx: tuple[int, int, int],   # (patch_D, patch_H, patch_W)
    sampling_mode: str,
    foreground_class: int,
    number_of_samples: int,
    generator: torch.Generator,
) -> tuple[int, float]:                     # (hit_count, hit_rate)
    """Sampling policy별 foreground patch 포함 비율 측정"""
    
    foreground_hit_count = 0
    
    for _ in range(number_of_samples):
        # Sampling mode에 따른 patch center 선택
        if sampling_mode == "uniform":
            sampled_center = sample_uniform_center(
                volume_shape_zyx=label_volume.shape[-3:],
                generator=generator,
            )

        elif sampling_mode == "foreground":
            sampled_center = sample_foreground_center(
                label_volume=label_volume,
                foreground_class=foreground_class,
                generator=generator,
            )

        else:
            raise ValueError(
                f"지원하지 않는 sampling mode: {sampling_mode}"
            )
            
        # 선택된 center 기준 label patch 추출
        sampled_patch = crop_3d_patch_with_padding(
            volume=label_volume,            # [B, 1, D, H, W]
            center_zyx=sampled_center,
            patch_size_zyx=patch_size_zyx,
            padding_value=0.0,
        )                                   # [B, 1, patch_D, patch_H, patch_W]

        # Patch 내부의 foreground class 존재 여부 판정
        contains_foreground = (
            sampled_patch == foreground_class
        ).any().item()

        # Foreground를 포함한 patch 개수 누적
        if contains_foreground:
            foreground_hit_count += 1
            
    # 전체 sampling 횟수 중 foreground patch 비율 계산
    foreground_hit_rate = (
        foreground_hit_count
        / number_of_samples
    )

    return foreground_hit_count, foreground_hit_rate


# Cell을 실행할 때마다 새로운 comparison seed 생성
seed_source = torch.Generator()
comparison_seed = seed_source.seed()

# 두 policy의 결과를 재현할 수 있도록 seed 기록
print("Comparison seed:", comparison_seed)
print()

# 두 sampling policy에 독립적인 Generator 생성
uniform_generator = torch.Generator()
uniform_generator.manual_seed(
    comparison_seed,
)

foreground_generator = torch.Generator()
foreground_generator.manual_seed(
    comparison_seed,
)

number_of_trials = 1000

# Volume 전체에서 center를 선택하는 uniform policy 측정
uniform_hit_count, uniform_hit_rate = (
    measure_foreground_patch_hit_rate(
        label_volume=synthetic_label,       # [1, 1, 32, 64, 64]
        patch_size_zyx=uniform_patch_size,
        sampling_mode="uniform",
        foreground_class=1,
        number_of_samples=number_of_trials,
        generator=uniform_generator,
    )
)

# Class 1 voxel에서 center를 선택하는 foreground policy 측정
foreground_hit_count, foreground_hit_rate = (
    measure_foreground_patch_hit_rate(
        label_volume=synthetic_label,       # [1, 1, 32, 64, 64]
        patch_size_zyx=uniform_patch_size,
        sampling_mode="foreground",
        foreground_class=1,
        number_of_samples=number_of_trials,
        generator=foreground_generator,
    )
)

# 두 policy의 foreground 노출 빈도 차이 계산
hit_rate_difference = (
    foreground_hit_rate
    - uniform_hit_rate
)

print(
    "Uniform sampling:   ",
    f"{uniform_hit_count}/{number_of_trials}",
    f"({uniform_hit_rate:.1%})",
)

print(
    "Foreground sampling:",
    f"{foreground_hit_count}/{number_of_trials}",
    f"({foreground_hit_rate:.1%})",
)

print(
    "Hit-rate difference:",
    f"{hit_rate_difference:.1%}p",
)

Comparison seed: 3729705029585035927

Uniform sampling:    43/1000 (4.3%)
Foreground sampling: 1000/1000 (100.0%)
Hit-rate difference: 95.7%p
